# 🎬 Movie Recommendation System Using Collaborative Filtering

### Artificial Intelligence — Course Project

**Project Type:** Machine Learning / Recommendation System
**Technique Used:** User-Based Collaborative Filtering (Cosine Similarity)
**Dataset:** MovieLens Latest Small Dataset (ml-latest-small)

---

## 📑 Table of Contents

1. [Project Introduction](#section1)
2. [Installation & Import Libraries](#section2)
3. [Dataset Loading](#section3)
4. [Exploratory Data Analysis (EDA)](#section4)
5. [Data Preprocessing](#section5)
6. [User-Item Matrix](#section6)
7. [Collaborative Filtering](#section7)
8. [Recommendation Function](#section8)
9. [Model Evaluation](#section9)
10. [Demonstration](#section10)
11. [Results & Discussion](#section11)
12. [Conclusion](#section12)
13. [Bonus Features](#bonus)
14. [References](#references)

---

<a id="section1"></a>
## 1. 📌 Project Introduction

### 1.1 What is a Recommendation System?

A **Recommendation System** (or Recommender System) is an intelligent software tool that
predicts and suggests items (movies, products, songs, courses, etc.) that a user is most
likely to be interested in, based on historical data such as past behavior, preferences,
and interactions with other users or items.

Recommendation systems solve the **information overload** problem — instead of showing a
user millions of available items, the system narrows it down to a personalized, ranked
shortlist that maximizes relevance and engagement.

Formally, given a set of users `U`, a set of items `I`, and a (typically sparse) set of
known user-item interactions (ratings, clicks, purchases), a recommendation system tries to
estimate a utility function:

```
f : U x I -> R
```

that predicts how much a user `u` would like an item `i`, and then recommends the items
with the highest predicted utility that the user has not yet interacted with.

### 1.2 Types of Recommendation Systems

**a) Content-Based Filtering**
- Recommends items similar to those the user has liked in the past.
- Uses item attributes/features (genre, actors, director, description, keywords, etc.)
- Builds a profile of user preferences and matches it against item feature vectors.
- Limitation: Cannot capture preferences that are not explained by item attributes
  (it tends to over-specialize and creates a "filter bubble").

**b) Collaborative Filtering (CF)** — *Used in this project*
- Recommends items based on the preferences of **similar users** or **similar items**,
  without needing any information about the content of the items themselves.
- Two major approaches:
  - **User-Based CF:** "Users who are similar to you also liked these movies."
  - **Item-Based CF:** "Users who liked this movie also liked these other movies."
- Relies purely on the user-item interaction matrix (ratings).
- Strength: Can discover unexpected/serendipitous recommendations.
- Weakness: Suffers from the *cold-start problem* (new users/items with no ratings) and
  *sparsity* (most users rate only a tiny fraction of all items).

**c) Hybrid Recommendation Systems**
- Combine content-based and collaborative filtering (and sometimes knowledge-based or
  demographic approaches) to overcome the individual weaknesses of each method.
- Example: Netflix and YouTube use hybrid systems that combine collaborative signals,
  content metadata, and deep learning-based contextual signals.

### 1.3 Real-World Applications of Recommendation Systems

| Platform   | Use Case                                                          |
|------------|--------------------------------------------------------------------|
| Netflix    | Recommends movies/TV shows based on viewing history & similar users |
| Amazon     | "Customers who bought this also bought…" product recommendations   |
| YouTube    | Suggests videos based on watch history and similar viewers          |
| Spotify    | Recommends songs/playlists (Discover Weekly) using listening habits  |
| Coursera   | Suggests courses based on learning history and peer behavior         |
| Udemy      | Recommends courses similar students have purchased/completed         |

### 1.4 Project Architecture Diagram

```
                ┌─────────────────────────┐
                │   MovieLens Dataset     │
                │  (ratings.csv, movies.csv) │
                └───────────┬─────────────┘
                            │
                            ▼
                ┌─────────────────────────┐
                │   Data Preprocessing    │
                │  - Clean / Merge / Dedup │
                └───────────┬─────────────┘
                            │
                            ▼
                ┌─────────────────────────┐
                │     User-Item Matrix    │
                │ rows=users, cols=movies │
                └───────────┬─────────────┘
                            │
                            ▼
                ┌─────────────────────────┐
                │  Cosine Similarity      │
                │  (User-User Similarity) │
                └───────────┬─────────────┘
                            │
                            ▼
                ┌─────────────────────────┐
                │ Collaborative Filtering │
                │  Rating Prediction      │
                └───────────┬─────────────┘
                            │
                            ▼
                ┌─────────────────────────┐
                │  recommend_movies()     │
                │   Top-N Recommendations │
                └───────────┬─────────────┘
                            │
                            ▼
                ┌─────────────────────────┐
                │   Model Evaluation      │
                │     (RMSE, MAE)         │
                └─────────────────────────┘
```

### 1.5 Project Objectives

1. Load and explore the MovieLens Latest Small dataset.
2. Perform Exploratory Data Analysis (EDA) to understand rating patterns.
3. Build a User-Item rating matrix.
4. Implement User-Based Collaborative Filtering using Cosine Similarity.
5. Build a `recommend_movies()` function that returns Top-N personalized recommendations.
6. Evaluate the model using RMSE and MAE.
7. Demonstrate recommendations for multiple sample users.
8. Provide bonus utilities (search, trending, top-rated, interactive lookup).

<a id="section2"></a>
## 2. ⚙️ Installation & Import Libraries

The cell below installs (if required) and imports all the libraries used throughout this
notebook. This notebook is compatible with both **Jupyter Notebook** and **Google Colab**.

- `pandas`, `numpy` → data loading & numerical operations
- `matplotlib`, `seaborn` → data visualization
- `scikit-learn` → cosine similarity, RMSE, MAE metrics

In [ ]:
# ----------------------------------------------------------------------
# Uncomment the line below if running on Google Colab or a fresh
# environment that does not already have these packages installed.
# ----------------------------------------------------------------------
# !pip install pandas numpy matplotlib seaborn scikit-learn --quiet

# Core data handling libraries
import pandas as pd                     # for dataframes and data manipulation
import numpy as np                      # for numerical/array operations

# Visualization libraries
import matplotlib.pyplot as plt         # for plotting charts
import seaborn as sns                   # for advanced/statistical visualizations

# Machine learning utilities
from sklearn.metrics.pairwise import cosine_similarity   # to compute user-user similarity
from sklearn.metrics import mean_squared_error           # for RMSE evaluation
from sklearn.metrics import mean_absolute_error          # for MAE evaluation
from sklearn.model_selection import train_test_split      # to split ratings for evaluation

import os               # file/path handling
import zipfile          # to extract the downloaded dataset
import urllib.request   # to download the dataset automatically
import warnings

warnings.filterwarnings("ignore")  # suppress non-critical warnings for a clean output

# Set a consistent, professional visual style for all plots in this notebook
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"

print("✅ All libraries imported successfully.")

<a id="section3"></a>
## 3. 📂 Dataset Loading

**Dataset:** MovieLens Latest Small Dataset (`ml-latest-small`)

- Official source: https://grouplens.org/datasets/movielens/latest/
- Mirror / alternative (Kaggle): https://www.kaggle.com/datasets/grouplens/movielens-latest-small

This dataset contains:
- **`ratings.csv`** → `userId, movieId, rating, timestamp`
- **`movies.csv`** → `movieId, title, genres`

The code below automatically downloads and extracts the dataset directly from the official
GroupLens server. If no internet connection is available, simply download the ZIP file
manually from one of the links above, extract it, and place the `ml-latest-small` folder in
the same directory as this notebook.

In [ ]:
# ----------------------------------------------------------------------
# Automatically download and extract the MovieLens Latest Small dataset
# ----------------------------------------------------------------------
DATASET_URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
DATASET_DIR = "ml-latest-small"
ZIP_PATH = "ml-latest-small.zip"

if not os.path.exists(DATASET_DIR):
    try:
        print("Downloading MovieLens Latest Small dataset...")
        urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)        # download the zip file
        with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
            zip_ref.extractall(".")                              # extract into current directory
        print("✅ Dataset downloaded and extracted successfully.")
    except Exception as e:
        # If download fails (e.g. no internet), instruct the user to download manually.
        print("⚠️ Automatic download failed:", e)
        print("Please manually download the dataset from:")
        print("https://grouplens.org/datasets/movielens/latest/")
        print("and extract the 'ml-latest-small' folder next to this notebook.")
else:
    print("Dataset folder already exists locally. Skipping download.")

In [ ]:
# ----------------------------------------------------------------------
# Load the ratings and movies CSV files into pandas DataFrames
# ----------------------------------------------------------------------
ratings = pd.read_csv(os.path.join(DATASET_DIR, "ratings.csv"))   # user-movie ratings
movies = pd.read_csv(os.path.join(DATASET_DIR, "movies.csv"))     # movie metadata (title, genres)

print("Ratings dataset and Movies dataset loaded successfully.\n")

In [ ]:
# ----------------------------------------------------------------------
# Preview the first few rows of each dataset
# ----------------------------------------------------------------------
print("RATINGS DATASET — first 5 rows:")
display(ratings.head())

print("\nMOVIES DATASET — first 5 rows:")
display(movies.head())

In [ ]:
# ----------------------------------------------------------------------
# Check the shape (rows, columns) of each dataset
# ----------------------------------------------------------------------
print("Shape of ratings dataset:", ratings.shape)
print("Shape of movies dataset :", movies.shape)

In [ ]:
# ----------------------------------------------------------------------
# Display structural information (data types, non-null counts, memory)
# ----------------------------------------------------------------------
print("RATINGS DATASET INFO:")
ratings.info()

print("\nMOVIES DATASET INFO:")
movies.info()

In [ ]:
# ----------------------------------------------------------------------
# Display statistical summary of the numerical columns
# ----------------------------------------------------------------------
print("Statistical Summary of Ratings Dataset:")
display(ratings.describe())

In [ ]:
# ----------------------------------------------------------------------
# Check for missing (null) values in both datasets
# ----------------------------------------------------------------------
print("Missing values in RATINGS dataset:")
print(ratings.isnull().sum())

print("\nMissing values in MOVIES dataset:")
print(movies.isnull().sum())

In [ ]:
# ----------------------------------------------------------------------
# Check for duplicate rows in both datasets
# ----------------------------------------------------------------------
print("Duplicate rows in RATINGS dataset:", ratings.duplicated().sum())
print("Duplicate rows in MOVIES dataset :", movies.duplicated().sum())

<a id="section4"></a>
## 4. 📊 Exploratory Data Analysis (EDA)

In this section we explore the dataset to understand:

1. Number of unique users
2. Number of unique movies
3. Total number of ratings
4. Average rating across the dataset
5. Top rated movies (highest average rating)
6. Most rated movies (highest number of ratings)

We then visualize the distributions using histograms, bar charts, pie charts, and heatmaps.

In [ ]:
# ----------------------------------------------------------------------
# 4.1 Basic dataset statistics
# ----------------------------------------------------------------------
num_users = ratings["userId"].nunique()          # count of unique users
num_movies = ratings["movieId"].nunique()        # count of unique movies rated
total_ratings = ratings.shape[0]                 # total number of rating records
average_rating = ratings["rating"].mean()        # average rating value

print(f"👤 Number of Unique Users   : {num_users}")
print(f"🎬 Number of Unique Movies  : {num_movies}")
print(f"⭐ Total Ratings Given      : {total_ratings}")
print(f"📈 Average Rating (overall) : {average_rating:.3f}")

In [ ]:
# ----------------------------------------------------------------------
# 4.2 Top Rated Movies (highest average rating, with a minimum vote count
#     threshold so that movies rated only once or twice don't dominate)
# ----------------------------------------------------------------------
MIN_VOTES = 50   # minimum number of ratings required to qualify as "top rated"

movie_stats = ratings.groupby("movieId")["rating"].agg(["mean", "count"]).reset_index()
movie_stats.columns = ["movieId", "avg_rating", "rating_count"]
movie_stats = movie_stats.merge(movies[["movieId", "title"]], on="movieId", how="left")

top_rated_movies = (
    movie_stats[movie_stats["rating_count"] >= MIN_VOTES]
    .sort_values("avg_rating", ascending=False)
    .head(10)
)

print(f"🏆 TOP 10 RATED MOVIES (with at least {MIN_VOTES} ratings):")
display(top_rated_movies[["title", "avg_rating", "rating_count"]])

In [ ]:
# ----------------------------------------------------------------------
# 4.3 Most Rated Movies (highest number of ratings, regardless of score)
# ----------------------------------------------------------------------
most_rated_movies = movie_stats.sort_values("rating_count", ascending=False).head(10)

print("🔥 TOP 10 MOST RATED MOVIES:")
display(most_rated_movies[["title", "rating_count", "avg_rating"]])

In [ ]:
# ----------------------------------------------------------------------
# 4.4 Visualization 1: Histogram — Distribution of Ratings
# ----------------------------------------------------------------------
plt.figure(figsize=(8, 5))
sns.histplot(ratings["rating"], bins=10, kde=False, color="royalblue", edgecolor="black")
plt.title("Distribution of Movie Ratings", fontsize=14, fontweight="bold")
plt.xlabel("Rating Value")
plt.ylabel("Number of Ratings")
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------------------------------------
# 4.5 Visualization 2: Bar Chart — Top 10 Most Rated Movies
# ----------------------------------------------------------------------
plt.figure(figsize=(10, 6))
sns.barplot(
    data=most_rated_movies,
    x="rating_count",
    y="title",
    hue="title",
    palette="viridis",
    legend=False,
)
plt.title("Top 10 Most Rated Movies", fontsize=14, fontweight="bold")
plt.xlabel("Number of Ratings")
plt.ylabel("Movie Title")
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------------------------------------
# 4.6 Visualization 3: Pie Chart — Proportion of each Rating Value
# ----------------------------------------------------------------------
rating_counts = ratings["rating"].value_counts().sort_index()

plt.figure(figsize=(8, 8))
plt.pie(
    rating_counts.values,
    labels=[f"{r} ⭐" for r in rating_counts.index],
    autopct="%1.1f%%",
    startangle=90,
    colors=sns.color_palette("Set3", len(rating_counts)),
)
plt.title("Proportion of Each Rating Value", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------------------------------------
# 4.7 Visualization 4: Heatmap — Average Rating by Genre vs Rating Bucket
# ----------------------------------------------------------------------
# Split the pipe-separated genre string into individual genre rows
movies_genres = movies.copy()
movies_genres["genres"] = movies_genres["genres"].str.split("|")
movies_exploded = movies_genres.explode("genres")

genre_ratings = movies_exploded.merge(ratings, on="movieId", how="inner")

# Build a pivot table: average rating per genre
genre_avg = (
    genre_ratings.groupby("genres")["rating"]
    .mean()
    .sort_values(ascending=False)
    .head(15)
    .to_frame()
)

plt.figure(figsize=(6, 8))
sns.heatmap(genre_avg, annot=True, fmt=".2f", cmap="YlGnBu", cbar=True)
plt.title("Average Rating by Movie Genre (Top 15)", fontsize=14, fontweight="bold")
plt.xlabel("")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

<a id="section5"></a>
## 5. 🧹 Data Preprocessing

Before building the recommendation model, the data needs to be cleaned and prepared:

1. **Missing Value Handling** — verify and handle any nulls.
2. **Duplicate Removal** — remove any duplicate rating records.
3. **Data Cleaning** — ensure correct data types.
4. **Merging** — combine `ratings` and `movies` into a single working dataset.
5. **Movie-Rating Dataset** — an aggregated dataset summarizing each movie's rating stats.

In [ ]:
# ----------------------------------------------------------------------
# 5.1 Handle missing values (defensive coding — dataset is normally clean)
# ----------------------------------------------------------------------
ratings = ratings.dropna()    # drop any rows with missing values, if present
movies = movies.dropna(subset=["title"])  # a movie must at least have a title

print("Missing values remaining in ratings:", ratings.isnull().sum().sum())
print("Missing values remaining in movies :", movies.isnull().sum().sum())

In [ ]:
# ----------------------------------------------------------------------
# 5.2 Remove duplicate rows
# ----------------------------------------------------------------------
before_ratings = ratings.shape[0]
ratings = ratings.drop_duplicates()
after_ratings = ratings.shape[0]

before_movies = movies.shape[0]
movies = movies.drop_duplicates()
after_movies = movies.shape[0]

print(f"Ratings: removed {before_ratings - after_ratings} duplicate rows.")
print(f"Movies : removed {before_movies - after_movies} duplicate rows.")

In [ ]:
# ----------------------------------------------------------------------
# 5.3 Data type cleaning — ensure correct dtypes for merging/pivoting
# ----------------------------------------------------------------------
ratings["userId"] = ratings["userId"].astype(int)
ratings["movieId"] = ratings["movieId"].astype(int)
ratings["rating"] = ratings["rating"].astype(float)
movies["movieId"] = movies["movieId"].astype(int)

print("Data types after cleaning:")
print(ratings.dtypes)
print(movies.dtypes)

In [ ]:
# ----------------------------------------------------------------------
# 5.4 Merge ratings and movies into a single combined dataset
# ----------------------------------------------------------------------
movie_ratings = ratings.merge(movies, on="movieId", how="left")

print("Merged Movie-Rating Dataset (first 5 rows):")
display(movie_ratings.head())
print("\nShape of merged dataset:", movie_ratings.shape)

In [ ]:
# ----------------------------------------------------------------------
# 5.5 Build an aggregated Movie-Rating summary dataset
#     (one row per movie, with rating count and average rating)
# ----------------------------------------------------------------------
movie_rating_summary = (
    movie_ratings.groupby(["movieId", "title", "genres"])["rating"]
    .agg(rating_count="count", avg_rating="mean")
    .reset_index()
    .sort_values("rating_count", ascending=False)
)

print("Aggregated Movie-Rating Summary Dataset (first 10 rows):")
display(movie_rating_summary.head(10))

<a id="section6"></a>
## 6. 🧮 User-Item Matrix

A **User-Item Matrix** is the foundation of Collaborative Filtering. It is a matrix where:

- **Rows** represent users (`userId`)
- **Columns** represent movies (`movieId`)
- **Values** represent the rating a user gave to a movie (0 / NaN if not rated)

This matrix is naturally very **sparse**, since each user typically rates only a small
fraction of all available movies.

In [ ]:
# ----------------------------------------------------------------------
# 6.1 Build the User-Item Matrix using a pivot table
# ----------------------------------------------------------------------
user_item_matrix = ratings.pivot_table(index="userId", columns="movieId", values="rating")

# Fill unrated movies with 0 (0 simply means "not rated", not an actual low score)
user_item_matrix_filled = user_item_matrix.fillna(0)

print("User-Item Matrix dimensions (users x movies):", user_item_matrix.shape)
display(user_item_matrix.iloc[:5, :8])   # preview a small slice of the matrix

In [ ]:
# ----------------------------------------------------------------------
# 6.2 Calculate and visualize the sparsity of the User-Item Matrix
# ----------------------------------------------------------------------
total_cells = user_item_matrix.shape[0] * user_item_matrix.shape[1]
filled_cells = ratings.shape[0]
sparsity = 1 - (filled_cells / total_cells)

print(f"Total possible (user, movie) pairs : {total_cells}")
print(f"Actual ratings provided            : {filled_cells}")
print(f"Matrix Sparsity                    : {sparsity * 100:.3f} %")

plt.figure(figsize=(6, 6))
plt.pie(
    [filled_cells, total_cells - filled_cells],
    labels=["Rated", "Unrated"],
    autopct="%1.2f%%",
    colors=["#4CAF50", "#E0E0E0"],
    startangle=90,
)
plt.title("User-Item Matrix Sparsity", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------------------------------------
# 6.3 Heatmap preview of a small sample of the User-Item Matrix
# ----------------------------------------------------------------------
sample_matrix = user_item_matrix_filled.iloc[:25, :25]   # first 25 users x 25 movies

plt.figure(figsize=(12, 8))
sns.heatmap(sample_matrix, cmap="YlOrRd", cbar=True, linewidths=0.2)
plt.title("Sample of User-Item Matrix (25 Users x 25 Movies)", fontsize=14, fontweight="bold")
plt.xlabel("Movie ID")
plt.ylabel("User ID")
plt.tight_layout()
plt.show()

<a id="section7"></a>
## 7. 🤝 Collaborative Filtering (User-Based)

**User-Based Collaborative Filtering** recommends movies to a user based on the preferences
of *other users who have similar taste*.

### Steps:
1. Compute **Cosine Similarity** between every pair of users using their rating vectors.
2. For a target user, find the **most similar users** (nearest neighbors).
3. Use the ratings of those similar users to **predict ratings** for movies the target user
   has not yet seen, and generate the final recommendations.

### Cosine Similarity Formula

For two users `u` and `v` with rating vectors `Ru` and `Rv`:

```
                 Ru . Rv
sim(u, v) = ----------------------
             ||Ru|| * ||Rv||
```

A similarity value close to **1** means the two users rate movies very similarly; a value
close to **0** means they have little in common.

In [ ]:
# ----------------------------------------------------------------------
# 7.1 Compute User-User Cosine Similarity Matrix
# ----------------------------------------------------------------------
similarity_matrix = cosine_similarity(user_item_matrix_filled)

# Wrap the similarity matrix in a labeled DataFrame for easy lookup
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=user_item_matrix_filled.index,
    columns=user_item_matrix_filled.index,
)

print("User-User Similarity Matrix shape:", similarity_df.shape)
display(similarity_df.iloc[:8, :8])   # preview similarity scores for the first 8 users

In [ ]:
# ----------------------------------------------------------------------
# 7.2 Visualize the Similarity Matrix (sample of 30 users) as a heatmap
# ----------------------------------------------------------------------
plt.figure(figsize=(10, 8))
sns.heatmap(similarity_df.iloc[:30, :30], cmap="coolwarm", cbar=True)
plt.title("User-User Cosine Similarity (Sample of 30 Users)", fontsize=14, fontweight="bold")
plt.xlabel("User ID")
plt.ylabel("User ID")
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------------------------------------
# 7.3 Helper: Find the Top-K most similar users to a given user
# ----------------------------------------------------------------------
def get_similar_users(user_id, k=10):
    """Return the k most similar users to user_id, excluding the user itself."""
    if user_id not in similarity_df.index:
        raise ValueError(f"User {user_id} not found in the dataset.")

    # sort other users by similarity score in descending order
    similar_scores = similarity_df[user_id].drop(index=user_id).sort_values(ascending=False)
    return similar_scores.head(k)


# Example: show the 10 most similar users to User 1
example_similar_users = get_similar_users(1, k=10)
print("Top 10 users most similar to User 1:")
display(example_similar_users.to_frame(name="similarity_score"))

<a id="section8"></a>
## 8. 🎯 Recommendation Function

We now combine everything into two core functions:

1. **`predict_rating(user_id, movie_id, ...)`** — predicts the rating a user would give to a
   movie they have not yet seen, using a **mean-centered, similarity-weighted average** of
   the ratings given by the user's most similar neighbors.

2. **`recommend_movies(user_id, top_n=10)`** — uses `predict_rating()` to score all unseen
   candidate movies (movies rated by the user's nearest neighbors) and returns the
   **Top-N highest predicted** movies as a clean DataFrame.

### Prediction Formula (Mean-Centered User-Based CF)

```
                              Σ sim(u, v) * (r(v, i) - mean(v))
pred(u, i) = mean(u) +   --------------------------------------
                                    Σ |sim(u, v)|
```

Where the sum is taken over the `k` most similar users `v` who have rated movie `i`.

In [ ]:
# ----------------------------------------------------------------------
# 8.1 Pre-compute each user's average rating (used for mean-centering)
# ----------------------------------------------------------------------
user_mean_ratings = ratings.groupby("userId")["rating"].mean()
GLOBAL_MEAN_RATING = ratings["rating"].mean()

print("Global average rating across all users:", round(GLOBAL_MEAN_RATING, 3))

In [ ]:
# ----------------------------------------------------------------------
# 8.2 predict_rating() — predicts a single user-movie rating
# ----------------------------------------------------------------------
def predict_rating(user_id, movie_id, ui_matrix, sim_df, k=20):
    """
    Predict the rating that `user_id` would give to `movie_id`
    using mean-centered, similarity-weighted User-Based Collaborative Filtering.
    """
    # If the movie doesn't exist in the matrix, fall back to the global mean
    if movie_id not in ui_matrix.columns:
        return GLOBAL_MEAN_RATING

    # If the user doesn't exist, fall back to the global mean
    if user_id not in ui_matrix.index:
        return GLOBAL_MEAN_RATING

    # Get similarity scores of all other users to this user
    sims = sim_df[user_id].drop(index=user_id, errors="ignore")

    # Keep only users who have actually rated this movie (rating > 0)
    raters = ui_matrix.index[ui_matrix[movie_id] > 0]
    sims = sims[sims.index.isin(raters)]

    if sims.empty:
        # No similar user has rated this movie -> fall back to user's own average
        return user_mean_ratings.get(user_id, GLOBAL_MEAN_RATING)

    # Keep only the top-k most similar neighbors among the raters
    top_k_sims = sims.sort_values(ascending=False).head(k)

    numerator = 0.0
    denominator = 0.0
    user_avg = user_mean_ratings.get(user_id, GLOBAL_MEAN_RATING)

    for neighbor_id, sim_score in top_k_sims.items():
        neighbor_rating = ui_matrix.loc[neighbor_id, movie_id]
        neighbor_avg = user_mean_ratings.get(neighbor_id, GLOBAL_MEAN_RATING)
        numerator += sim_score * (neighbor_rating - neighbor_avg)
        denominator += abs(sim_score)

    if denominator == 0:
        return user_avg

    predicted = user_avg + (numerator / denominator)

    # Clip the prediction to the valid MovieLens rating range [0.5, 5.0]
    return float(np.clip(predicted, 0.5, 5.0))


# Quick test: predict a rating for User 1 on a movie they have already rated
test_movie_id = user_item_matrix.columns[0]
print(f"Predicted rating for User 1 on Movie {test_movie_id}:",
      round(predict_rating(1, test_movie_id, user_item_matrix_filled, similarity_df), 2))

In [ ]:
# ----------------------------------------------------------------------
# 8.3 recommend_movies() — generates Top-N recommendations for a user
# ----------------------------------------------------------------------
def recommend_movies(user_id, top_n=10, k=20):
    """
    Generate the Top-N movie recommendations for a given user using
    User-Based Collaborative Filtering.

    Steps:
      1. Find the k most similar users to `user_id`.
      2. Collect all movies those similar users have rated (candidate movies).
      3. Remove movies the target user has already seen.
      4. Predict a rating for each remaining candidate movie.
      5. Return the Top-N highest predicted movies.
    """
    if user_id not in user_item_matrix_filled.index:
        print(f"⚠️ User {user_id} does not exist in the dataset.")
        return pd.DataFrame(columns=["movieId", "title", "genres", "predicted_rating"])

    # Step 1: find similar users
    similar_users = get_similar_users(user_id, k=k)

    # Step 2: gather candidate movies rated by those similar users
    neighbors_ratings = user_item_matrix_filled.loc[similar_users.index]
    candidate_movie_ids = neighbors_ratings.columns[(neighbors_ratings > 0).any(axis=0)]

    # Step 3: exclude movies the target user has already rated/seen
    user_seen_mask = user_item_matrix_filled.loc[user_id] > 0
    already_seen = set(user_item_matrix_filled.columns[user_seen_mask])
    unseen_candidates = [m for m in candidate_movie_ids if m not in already_seen]

    # Step 4: predict a rating for every unseen candidate movie
    predictions = [
        (movie_id, predict_rating(user_id, movie_id, user_item_matrix_filled, similarity_df, k=k))
        for movie_id in unseen_candidates
    ]

    if not predictions:
        return pd.DataFrame(columns=["movieId", "title", "genres", "predicted_rating"])

    # Step 5: build a results DataFrame and keep the Top-N highest predicted ratings
    results_df = pd.DataFrame(predictions, columns=["movieId", "predicted_rating"])
    results_df = results_df.merge(movies, on="movieId", how="left")
    results_df = results_df.sort_values("predicted_rating", ascending=False).head(top_n)
    results_df = results_df[["movieId", "title", "genres", "predicted_rating"]].reset_index(drop=True)

    return results_df


# Example usage: Top-10 recommendations for User 1
recommend_movies(1, top_n=10)

<a id="section9"></a>
## 9. 📐 Model Evaluation

To objectively measure how good our Collaborative Filtering model is, we:

1. Split the ratings dataset into a **training set (80%)** and a **test set (20%)**.
2. Rebuild the User-Item Matrix and Similarity Matrix using only the **training** data.
3. For every rating in the **test set**, predict it using `predict_rating()` and compare it
   against the actual rating.
4. Compute the **RMSE (Root Mean Squared Error)** and **MAE (Mean Absolute Error)**.
5. Visualize Actual vs Predicted ratings.

In [ ]:
# ----------------------------------------------------------------------
# 9.1 Train/Test split of the ratings data
# ----------------------------------------------------------------------
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

print("Training set size:", train_data.shape[0])
print("Test set size    :", test_data.shape[0])

In [ ]:
# ----------------------------------------------------------------------
# 9.2 Rebuild the User-Item Matrix, Similarity Matrix, and user means
#     using ONLY the training data (to avoid data leakage)
# ----------------------------------------------------------------------
train_user_item_matrix = train_data.pivot_table(index="userId", columns="movieId", values="rating")
train_user_item_matrix_filled = train_user_item_matrix.fillna(0)

train_similarity_matrix = cosine_similarity(train_user_item_matrix_filled)
train_similarity_df = pd.DataFrame(
    train_similarity_matrix,
    index=train_user_item_matrix_filled.index,
    columns=train_user_item_matrix_filled.index,
)

train_user_mean_ratings = train_data.groupby("userId")["rating"].mean()
train_global_mean = train_data["rating"].mean()

print("Training User-Item Matrix shape  :", train_user_item_matrix_filled.shape)
print("Training Similarity Matrix shape :", train_similarity_df.shape)

In [ ]:
# ----------------------------------------------------------------------
# 9.3 Predict ratings for every (user, movie) pair in the test set
# ----------------------------------------------------------------------
def predict_rating_eval(user_id, movie_id, k=20):
    """Same logic as predict_rating(), but uses the TRAIN-only matrices."""
    if movie_id not in train_user_item_matrix_filled.columns or user_id not in train_user_item_matrix_filled.index:
        return train_global_mean

    sims = train_similarity_df[user_id].drop(index=user_id, errors="ignore")
    raters = train_user_item_matrix_filled.index[train_user_item_matrix_filled[movie_id] > 0]
    sims = sims[sims.index.isin(raters)]

    if sims.empty:
        return train_user_mean_ratings.get(user_id, train_global_mean)

    top_k_sims = sims.sort_values(ascending=False).head(k)
    user_avg = train_user_mean_ratings.get(user_id, train_global_mean)

    numerator, denominator = 0.0, 0.0
    for neighbor_id, sim_score in top_k_sims.items():
        neighbor_rating = train_user_item_matrix_filled.loc[neighbor_id, movie_id]
        neighbor_avg = train_user_mean_ratings.get(neighbor_id, train_global_mean)
        numerator += sim_score * (neighbor_rating - neighbor_avg)
        denominator += abs(sim_score)

    if denominator == 0:
        return user_avg

    return float(np.clip(user_avg + numerator / denominator, 0.5, 5.0))


# Generate predictions for the entire test set (may take a short while)
test_data = test_data.copy()
test_data["predicted_rating"] = test_data.apply(
    lambda row: predict_rating_eval(row["userId"], row["movieId"]), axis=1
)

display(test_data[["userId", "movieId", "rating", "predicted_rating"]].head(10))

In [ ]:
# ----------------------------------------------------------------------
# 9.4 Calculate RMSE and MAE
# ----------------------------------------------------------------------
actual = test_data["rating"]
predicted = test_data["predicted_rating"]

rmse = np.sqrt(mean_squared_error(actual, predicted))
mae = mean_absolute_error(actual, predicted)

print(f"📏 RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"📏 MAE  (Mean Absolute Error)    : {mae:.4f}")

In [ ]:
# ----------------------------------------------------------------------
# 9.5 Visualization: Actual vs Predicted Ratings (scatter plot)
# ----------------------------------------------------------------------
plt.figure(figsize=(8, 8))
plt.scatter(actual, predicted, alpha=0.3, color="teal", edgecolor="k", linewidth=0.2)
plt.plot([0.5, 5], [0.5, 5], color="red", linestyle="--", label="Perfect Prediction")
plt.xlabel("Actual Rating")
plt.ylabel("Predicted Rating")
plt.title("Actual vs Predicted Ratings", fontsize=14, fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------------------------------------
# 9.6 Visualization: Distribution comparison of Actual vs Predicted ratings
# ----------------------------------------------------------------------
plt.figure(figsize=(10, 6))
sns.kdeplot(actual, label="Actual Ratings", fill=True, color="royalblue", alpha=0.4)
sns.kdeplot(predicted, label="Predicted Ratings", fill=True, color="orange", alpha=0.4)
plt.title("Distribution of Actual vs Predicted Ratings", fontsize=14, fontweight="bold")
plt.xlabel("Rating")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

<a id="section10"></a>
## 10. 🚀 Demonstration

Here we generate live recommendations for three sample users — **User 1**, **User 50**, and
**User 100** — using the full model trained in Section 7 & 8 (built on the entire dataset).
For each user, the **Top-10 recommended movies** and their **predicted ratings** are shown.

In [ ]:
# ----------------------------------------------------------------------
# 10.1 Generate and display recommendations for sample users
# ----------------------------------------------------------------------
demo_user_ids = [1, 50, 100]

for uid in demo_user_ids:
    print("=" * 70)
    print(f"🎬 TOP 10 RECOMMENDATIONS FOR USER {uid}")
    print("=" * 70)

    recs = recommend_movies(uid, top_n=10)
    if recs.empty:
        print("No recommendations could be generated for this user.\n")
        continue

    display(recs[["title", "predicted_rating"]])
    print()

<a id="section11"></a>
## 11. 📋 Results & Discussion

### ✅ Advantages of User-Based Collaborative Filtering

- **No content/metadata required** — works purely from user-rating behavior, so it can
  recommend movies regardless of genre, language, or production details.
- **Serendipitous discovery** — can recommend items a user would not have searched for
  themselves, since recommendations come from the behavior of similar users.
- **Simple and interpretable** — cosine similarity and weighted averages are easy to
  explain to stakeholders ("recommended because similar users liked it").
- **Domain-independent** — the same technique generalizes to products, songs, courses, etc.

### ⚠️ Limitations

- **Cold-Start Problem** — new users (no ratings yet) or new movies (no ratings received)
  cannot be meaningfully recommended/recommended to.
- **Sparsity** — the User-Item matrix is extremely sparse (over 98% empty in this dataset),
  which can make similarity estimates noisy for users with very few ratings.
- **Scalability** — computing a full user-user similarity matrix is `O(n^2)` in the number
  of users, which becomes expensive for platforms with millions of users (e.g., Netflix).
- **Popularity Bias** — popular movies tend to get recommended more often simply because
  more users have rated them, sometimes overshadowing niche but relevant content.

### 🧩 Challenges Faced

- Handling the sparsity of the matrix while keeping predictions meaningful.
- Balancing the **k** (number of neighbors) to avoid overfitting to a tiny group of users
  versus diluting the signal with too many dissimilar neighbors.
- Efficient computation of recommendations without scanning the entire movie catalogue.

### 🔧 Possible Improvements

- Implement **Item-Based Collaborative Filtering** and compare performance.
- Use **Matrix Factorization** techniques (SVD, ALS) to handle sparsity more gracefully.
- Build a **Hybrid Recommender** combining collaborative filtering with movie genres/tags
  (content-based features) for better cold-start handling.
- Apply **deep learning** approaches (e.g., Neural Collaborative Filtering, Autoencoders).
- Incorporate **implicit feedback** (watch time, clicks) in addition to explicit ratings.
- Deploy the model behind a REST API / simple web app for real-time interactive use.

<a id="section12"></a>
## 12. 🏁 Conclusion

This project successfully implemented a **Movie Recommendation System** using **User-Based
Collaborative Filtering** on the **MovieLens Latest Small Dataset**.

### Project Objectives Recap
- ✅ Loaded and explored the MovieLens dataset (ratings + movies).
- ✅ Performed detailed Exploratory Data Analysis with multiple visualizations.
- ✅ Cleaned and merged the datasets, and constructed a User-Item ratings matrix.
- ✅ Computed user-user **Cosine Similarity** to identify users with similar tastes.
- ✅ Built a working `recommend_movies()` function producing Top-N personalized
  recommendations for any user.
- ✅ Evaluated the model quantitatively using **RMSE** and **MAE** on a held-out test set.
- ✅ Demonstrated the system end-to-end on three different users.

### Key Findings
- The dataset is highly sparse (~98%+ empty), which is typical of real-world recommender
  systems and directly motivates the need for collaborative filtering techniques.
- Mean-centered, similarity-weighted predictions perform reasonably well and produce
  sensible, personalized recommendations.
- Popularity and genre patterns strongly influence both rating behavior and the quality of
  similarity-based neighbor selection.

### Future Work
- Extend the system with **Item-Based CF**, **Matrix Factorization (SVD/ALS)**, and
  **Neural Collaborative Filtering** for improved accuracy and scalability.
- Combine collaborative and content-based signals into a **Hybrid Recommender**.
- Deploy the final model as an interactive web application (e.g., using Streamlit/Flask).

This notebook demonstrates a complete, end-to-end machine learning pipeline — from raw data
to a working, evaluated recommendation engine — suitable as a **Course Project**.

<a id="bonus"></a>
## 13. 🎁 Bonus Features

This section adds extra utility functions that make the recommender system more usable:

1. **`search_movie()`** — search for movies by (partial) title keyword.
2. **`top_trending_movies()`** — movies that have received the most ratings recently.
3. **`top_rated_movies()`** — highest average-rated movies (with a minimum vote threshold).
4. **Interactive User Input Section** — lets a user type in a `userId` and instantly get
   personalized recommendations.

In [ ]:
# ----------------------------------------------------------------------
# 13.1 Bonus Function: search_movie() — search movies by title keyword
# ----------------------------------------------------------------------
def search_movie(keyword, max_results=10):
    """Search the movies dataset for titles containing the given keyword (case-insensitive)."""
    keyword = keyword.strip().lower()
    matches = movies[movies["title"].str.lower().str.contains(keyword, na=False)]
    return matches.head(max_results).reset_index(drop=True)


# Example: search for movies containing the word "Batman"
print("Search results for 'Batman':")
display(search_movie("Batman"))

In [ ]:
# ----------------------------------------------------------------------
# 13.2 Bonus Function: top_trending_movies() — most recently popular movies
# ----------------------------------------------------------------------
def top_trending_movies(top_n=10, recent_fraction=0.2):
    """
    Identify 'trending' movies by looking at the most recent slice of ratings
    (based on timestamp) and finding which movies were rated most often within it.
    """
    cutoff = ratings["timestamp"].quantile(1 - recent_fraction)  # most recent X% of ratings
    recent_ratings = ratings[ratings["timestamp"] >= cutoff]

    trending = (
        recent_ratings.groupby("movieId")
        .agg(rating_count=("rating", "count"), avg_rating=("rating", "mean"))
        .reset_index()
        .merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
        .sort_values("rating_count", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    return trending[["title", "genres", "rating_count", "avg_rating"]]


print("🔥 Top 10 Trending Movies (recent activity):")
display(top_trending_movies(10))

In [ ]:
# ----------------------------------------------------------------------
# 13.3 Bonus Function: top_rated_movies() — best average rating overall
# ----------------------------------------------------------------------
def top_rated_movies(top_n=10, min_ratings=50):
    """Return the top-rated movies by average rating, requiring a minimum vote count."""
    qualified = movie_rating_summary[movie_rating_summary["rating_count"] >= min_ratings]
    return (
        qualified.sort_values("avg_rating", ascending=False)
        .head(top_n)[["title", "genres", "avg_rating", "rating_count"]]
        .reset_index(drop=True)
    )


print("🏆 Top 10 Rated Movies overall (min. 50 ratings):")
display(top_rated_movies(10))

In [ ]:
# ----------------------------------------------------------------------
# 13.4 Interactive User Input Section
# ----------------------------------------------------------------------
# This cell lets you type in any valid userId and instantly see their
# Top-N personalized movie recommendations. Run this cell in Jupyter
# Notebook / Google Colab and follow the prompt.
#
# NOTE: input() requires an interactive kernel. If running this notebook
# in a fully automated/non-interactive environment, this cell can be
# skipped — the function is still usable by calling it directly,
# e.g. interactive_recommendation_demo(user_id=42, top_n=5)
# ----------------------------------------------------------------------
def interactive_recommendation_demo(user_id=None, top_n=10):
    """Prompt for a userId (if not provided) and display their recommendations."""
    if user_id is None:
        try:
            user_id = int(input("Enter a userId to get movie recommendations: "))
        except (ValueError, EOFError):
            print("No valid input received. Using default User 1 instead.")
            user_id = 1

    print(f"\nGenerating Top-{top_n} recommendations for User {user_id}...\n")
    recs = recommend_movies(user_id, top_n=top_n)

    if recs.empty:
        print("Sorry, no recommendations could be generated for this user.")
    else:
        display(recs[["title", "genres", "predicted_rating"]])

    return recs


# Example non-interactive call (safe to run automatically):
interactive_recommendation_demo(user_id=25, top_n=5)

# To try it interactively yourself, uncomment the line below and run this cell:
# interactive_recommendation_demo()

<a id="references"></a>
## 14. 📚 Dataset Source & References

### Dataset Source

- **MovieLens Latest Small Dataset**
  - Official: https://grouplens.org/datasets/movielens/latest/
  - Alternative (Kaggle): https://www.kaggle.com/datasets/grouplens/movielens-latest-small
  - Direct download (used in this notebook): https://files.grouplens.org/datasets/movielens/ml-latest-small.zip

### References

1. F. Maxwell Harper and Joseph A. Konstan. 2015. *The MovieLens Datasets: History and
   Context*. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4, Article 19
   (December 2015), 19 pages. https://doi.org/10.1145/2827872
2. Goldberg, D., Nichols, D., Oki, B. M., & Terry, D. (1992). *Using collaborative filtering
   to weave an information tapestry*. Communications of the ACM, 35(12), 61-70.
3. Ricci, F., Rokach, L., & Shapira, B. (2015). *Recommender Systems Handbook* (2nd ed.).
   Springer.
4. Sarwar, B., Karypis, G., Konstan, J., & Riedl, J. (2001). *Item-based collaborative
   filtering recommendation algorithms*. Proceedings of the 10th International Conference on
   World Wide Web (WWW '01), 285-295.
5. scikit-learn documentation: https://scikit-learn.org/stable/modules/metrics.html#cosine-similarity
6. pandas documentation: https://pandas.pydata.org/docs/
7. seaborn documentation: https://seaborn.pydata.org/

---

### 📌 Project Summary

| Item                  | Detail                                                  |
|-----------------------|----------------------------------------------------------|
| Project Title         | Movie Recommendation System Using Collaborative Filtering |
| Technique              | User-Based Collaborative Filtering (Cosine Similarity)    |
| Dataset                | MovieLens Latest Small (ml-latest-small)                 |
| Evaluation Metrics     | RMSE, MAE                                                |
| Tools                  | Python, pandas, NumPy, scikit-learn, matplotlib, seaborn  |
| Project Type           | Course Project (Machine Learning / Recommender Systems) |

**— End of Notebook —**